# Week 43, Day 4: Brackets and undo

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w043d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
tokens = "3 4 + 2 *".split()
stack = []
for t in tokens:
    if t in "+-*/":
        b, a = stack.pop(), stack.pop()
        stack.append((a + b if t == "+" else a - b if t == "-" else a * b if t == "*" else a / b))
    else:
        stack.append(int(t))
    print(t, stack)

In [ ]:
PARTNER = {")": "(", "]": "[", "}": "{"}

def first_problem(text):
    """The position of the first bracket that's wrong, or -1 if they all match."""
    stack = []
    for i, ch in enumerate(text):
        if ch in "([{":
            stack.append((ch, i))
        elif ch in PARTNER:
            if not stack or stack[-1][0] != PARTNER[ch]:
                return i                     # a closer with no matching opener
            stack.pop()
    return stack[0][1] if stack else -1      # the earliest opener never closed

for text in ["f(a[1], {b: 2})", "f(a[1)]", "((x)", "x)"]:
    print(repr(text), first_problem(text))

In [ ]:
PRECEDENCE = {"+": 1, "-": 1, "*": 2, "/": 2, "^": 3}
RIGHT = {"^"}

def to_postfix(tokens):
    out, ops = [], []
    for t in tokens:
        if t in PRECEDENCE:
            while ops and ops[-1] != "(" and (PRECEDENCE[ops[-1]] > PRECEDENCE[t]
                                               or (PRECEDENCE[ops[-1]] == PRECEDENCE[t] and t not in RIGHT)):
                out.append(ops.pop())
            ops.append(t)
        elif t == "(":
            ops.append(t)
        elif t == ")":
            while ops[-1] != "(":
                out.append(ops.pop())
            ops.pop()                          # throw away the "("
        else:
            out.append(t)
    while ops:
        out.append(ops.pop())
    return out

for expr in ["3 + 4 * 2", "( 3 + 4 ) * 2", "10 - 2 - 3", "2 ^ 3 ^ 2", "1 + 2 * ( 3 - 4 ) / 5"]:
    print(f"{expr:<22} -> {' '.join(to_postfix(expr.split()))}")

In [ ]:
PRECEDENCE = {"+": 1, "-": 1, "*": 2, "/": 2}

def tokenise(text):
    tokens, number = [], ""
    for ch in text + " ":                       # a final space flushes the last number
        if ch.isdigit() or ch == ".":
            number += ch
            continue
        if number:
            tokens.append(number)
            number = ""
        if ch in "+-*/()":
            tokens.append(ch)
        elif not ch.isspace():
            raise ValueError(f"unexpected {ch!r}")
    return tokens

def to_postfix(tokens):
    out, ops = [], []
    for t in tokens:
        if t in PRECEDENCE:
            while ops and ops[-1] != "(" and PRECEDENCE[ops[-1]] >= PRECEDENCE[t]:
                out.append(ops.pop())
            ops.append(t)
        elif t == "(":
            ops.append(t)
        elif t == ")":
            while ops[-1] != "(":
                out.append(ops.pop())
            ops.pop()
        else:
            out.append(t)
    return out + ops[::-1]

def evaluate(postfix):
    stack = []
    for t in postfix:
        if t in PRECEDENCE:
            b, a = stack.pop(), stack.pop()
            stack.append((a + b if t == "+" else a - b if t == "-" else a * b if t == "*" else a / b))
        else:
            stack.append(float(t))
    return stack[0]

for text in ["2*(3+4.5)", "10 - 2 - 3", "7 / 2 + 1", "((1))"]:
    print(f"{text:<12} = {evaluate(to_postfix(tokenise(text)))}")

In [ ]:
from collections import deque

class Typist:
    def __init__(self, limit=100):
        self.text = ""
        self._undo = deque(maxlen=limit)     # each step: the text before it
        self._redo = []
        self._typing = False                 # is the last step a run of letters we can extend?

    def type(self, s):
        letters = s.isalpha()
        if not (letters and self._typing):
            self._undo.append(self.text)     # start a new step
        self._typing = letters
        self.text += s
        self._redo.clear()

    def undo(self):
        if not self._undo:
            return False
        self._redo.append(self.text)
        self.text = self._undo.pop()
        self._typing = False
        return True

    def redo(self):
        if not self._redo:
            return False
        self._undo.append(self.text)
        self.text = self._redo.pop()
        self._typing = False
        return True

t = Typist()
for piece in ["h", "e", "l", "l", "o", " ", "w", "o", "r", "l", "d"]:
    t.type(piece)
t.undo(); print(repr(t.text))
t.undo(); print(repr(t.text))
t.redo(); print(repr(t.text))

> **Key idea.** Nesting problems push openers (with positions) and check each closer against the top; leftovers at the end were never closed. Postfix puts operators after their operands, so one stack evaluates it: push numbers; an operator pops the second operand, then the first, and pushes the result. Dijkstra's shunting-yard converts infix to postfix with an operator stack: numbers go straight out; an operator first pops anything of higher precedence (or equal, if it's left-associative); `)` pops back to `(`. A calculator is a tokeniser, shunting-yard and an evaluator. Real undo keeps limited history (`deque(maxlen=...)`) and groups related actions into one step.

### Exercise: Where's the first problem?

Write `first_problem(text)`: the position of the first bracket that's wrong, or `-1` if all of `()[]{}` match and nest properly. A closer that doesn't match the most recent open bracket (or comes when none is open) is wrong at its own position; if the text ends with brackets still open, the answer is the position of the **earliest** one still open. Other characters are ignored.

In [ ]:
PARTNER = {")": "(", "]": "[", "}": "{"}

def first_problem(text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_problems():\n    \"problems\"\n    cases = {\"f(a[1], {b: 2})\": -1, \"\": -1, \"f(a[1)]\": 5, \"((x)\": 0, \"x)\": 1, \"([)]\": 2, \"(()\": 0, \"{[]}(\": 4, \"a(b)c]d\": 5}\n    for text, want in cases.items():\n        assert first_problem(text) == want, (text, first_problem(text))")

### Exercise: Nested tags

Write `check_tags(html)` for text with simple tags like `<b>`, `</b>` and `<br/>` (a self-closing tag, which needs no partner). Return `"ok"` if every tag is closed in the right order; otherwise `"unexpected </X>"` for a closing tag that doesn't match, or `"unclosed <X>"` for the **innermost** tag still open at the end. Tag names are compared in lower case. Anything between `<` and `>` is a tag; text outside tags is ignored.

In [ ]:
def check_tags(html):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_tags():\n    \"tags\"\n    cases = {\"<p>Hi <b>there</b></p>\": \"ok\", \"plain\": \"ok\", \"<p><b>x</p></b>\": \"unexpected </p>\", \"<p><i>x</i>\": \"unclosed <p>\",\n             \"<div><p>a<br/>b</p></div>\": \"ok\", \"</b>\": \"unexpected </b>\", \"<P>x</p>\": \"ok\", \"<ul><li>a<li>b</ul>\": \"unexpected </ul>\",\n             \"<a><b>\": \"unclosed <b>\"}\n    for html, want in cases.items():\n        assert check_tags(html) == want, (html, check_tags(html))")

### Exercise: Fewest brackets to add

Write `brackets_to_add(text)` for a string of `(` and `)` only: the fewest brackets that must be **added** (anywhere) to make it balanced. You don't need a real stack: a count of open brackets is a stack of identical items.

In [ ]:
def brackets_to_add(text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef slow(text):\n    s = text\n    while \"()\" in s:\n        s = s.replace(\"()\", \"\")\n    return len(s)\ndef test_add():\n    \"fewest\"\n    assert brackets_to_add(\"())\") == 1 and brackets_to_add(\"(((\") == 3 and brackets_to_add(\"()\") == 0 and brackets_to_add(\")(\") == 2 and brackets_to_add(\"\") == 0\n    rng = random.Random(1)\n    for _ in range(200):\n        t = \"\".join(rng.choice(\"()\") for _ in range(rng.randint(0, 12)))\n        assert brackets_to_add(t) == slow(t), t")

### Exercise: Evaluating postfix

Write `eval_postfix(tokens)` for a list of tokens: whole numbers (possibly negative, like `"-3"`) and the operators `+`, `-`, `*` and `/` (true division). Return the value. Raise `ValueError("too few operands")` if an operator finds fewer than two numbers, and `ValueError("leftover operands")` if more than one number is left at the end (or none at all).

In [ ]:
def eval_postfix(tokens):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_rpn():\n    \"postfix\"\n    assert eval_postfix(\"3 4 + 2 *\".split()) == 14 and eval_postfix(\"10 2 - 3 -\".split()) == 5 and eval_postfix(\"7 2 /\".split()) == 3.5\n    assert eval_postfix([\"-3\", \"4\", \"*\"]) == -12 and eval_postfix([\"5\"]) == 5\n    for tokens, message in [([\"1\", \"+\"], \"too few operands\"), ([\"1\", \"2\"], \"leftover operands\"), ([], \"leftover operands\")]:\n        try:\n            eval_postfix(tokens)\n            assert False, tokens\n        except ValueError as err:\n            assert str(err) == message")

### Exercise: Shunting-yard

Write `to_postfix(tokens)` with Dijkstra's shunting-yard algorithm, for numbers, `+ - * / ^` and brackets. `^` (power) has the highest precedence and is **right**-associative; the others are left-associative, with `* /` above `+ -`. Raise `ValueError("mismatched brackets")` for a `)` with no `(` or a `(` never closed.

In [ ]:
PRECEDENCE = {"+": 1, "-": 1, "*": 2, "/": 2, "^": 3}

def to_postfix(tokens):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_yard():\n    \"converting\"\n    cases = {\"3 + 4 * 2\": \"3 4 2 * +\", \"( 3 + 4 ) * 2\": \"3 4 + 2 *\", \"10 - 2 - 3\": \"10 2 - 3 -\", \"2 ^ 3 ^ 2\": \"2 3 2 ^ ^\",\n             \"1 + 2 * ( 3 - 4 ) / 5\": \"1 2 3 4 - * 5 / +\", \"8 / 4 / 2\": \"8 4 / 2 /\", \"2 * 3 ^ 2\": \"2 3 2 ^ *\", \"( ( 7 ) )\": \"7\"}\n    for expr, want in cases.items():\n        assert \" \".join(to_postfix(expr.split())) == want, (expr, to_postfix(expr.split()))\n    for bad in [\"( 1 + 2\", \"1 + 2 )\", \") (\"]:\n        try:\n            to_postfix(bad.split())\n            assert False, bad\n        except ValueError as err:\n            assert str(err) == \"mismatched brackets\"")

### Exercise: Undo with a limit and groups

Write `Editor(limit)` for a line of text, with `type(s)`, `delete(n)` (remove up to `n` characters from the end), `undo()` and `redo()` (both return `True` if they did something), and a `text` attribute. The undo history keeps at most `limit` steps (a `deque` with `maxlen`). Consecutive `type` calls whose text is all letters join one step; anything else (a non-letter typed, a delete, an undo or a redo) ends the group. Any new `type` or `delete` clears the redo stack.

In [ ]:
from collections import deque

class Editor:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_groups():\n    \"grouped steps\"\n    e = Editor(10)\n    for piece in [\"h\", \"i\", \" \", \"y\", \"o\", \"u\"]:\n        e.type(piece)\n    assert e.text == \"hi you\"\n    assert e.undo() and e.text == \"hi \" and e.undo() and e.text == \"hi\" and e.undo() and e.text == \"\" and not e.undo()\n    assert e.redo() and e.text == \"hi\" and e.redo() and e.redo() and e.text == \"hi you\" and not e.redo()\ndef test_delete():\n    \"delete and redo\"\n    e = Editor(10)\n    e.type(\"abc\")\n    e.delete(2)\n    assert e.text == \"a\" and e.undo() and e.text == \"abc\"\n    e.type(\"d\")\n    assert not e.redo(), \"a new action clears redo\"\n    e.delete(99)\n    assert e.text == \"\" and e.undo() and e.text == \"abcd\"\ndef test_limit():\n    \"limited history\"\n    e = Editor(3)\n    for piece in [\"a\", \"1\", \"b\", \"2\", \"c\"]:\n        e.type(piece)\n    count = 0\n    while e.undo():\n        count += 1\n    assert count == 3 and e.text == \"a1\"")

### Bug hunt: Operands the wrong way round

This postfix evaluator pops the operands into the wrong names, so subtraction and division come out backwards: `10 2 -` gives `-8` and `8 2 /` gives `0.25`. Fix it so this prints `8 4.0`.

In [ ]:
def eval_postfix(tokens):
    stack = []
    for t in tokens:
        if t in "+-*/":
            a, b = stack.pop(), stack.pop()
            stack.append((a + b if t == "+" else a - b if t == "-" else a * b if t == "*" else a / b))
        else:
            stack.append(int(t))
    return stack[0]

print(eval_postfix("10 2 -".split()), eval_postfix("8 2 /".split()))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the second operand is on top\"\n    expect_output(\"8 4.0\")")

### Bug hunt: Operators left behind

This shunting-yard forgets to pop the operators still on the stack at the end, so they never reach the output: this prints `3 4 2`. Fix it so it prints `3 4 2 * +`.

In [ ]:
PRECEDENCE = {"+": 1, "-": 1, "*": 2, "/": 2}

def to_postfix(tokens):
    out, ops = [], []
    for t in tokens:
        if t in PRECEDENCE:
            while ops and ops[-1] != "(" and PRECEDENCE[ops[-1]] >= PRECEDENCE[t]:
                out.append(ops.pop())
            ops.append(t)
        elif t == "(":
            ops.append(t)
        elif t == ")":
            while ops[-1] != "(":
                out.append(ops.pop())
            ops.pop()
        else:
            out.append(t)
    return out

print(" ".join(to_postfix("3 + 4 * 2".split())))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"flush the stack\"\n    expect_output(\"3 4 2 * +\")")

### Bug hunt: Never closed, never noticed

This tag checker checks every closing tag, but never checks whether tags are **still open** at the end, so unclosed tags pass: this prints `ok`. Fix it so it prints `unclosed <i>`.

In [ ]:
def check_tags(html):
    stack = []
    i = 0
    while html.find("<", i) != -1:
        start = html.find("<", i)
        end = html.find(">", start)
        name = html[start + 1:end].lower()
        i = end + 1
        if name.startswith("/"):
            if not stack or stack[-1] != name[1:]:
                return f"unexpected <{name}>"
            stack.pop()
        else:
            stack.append(name)
    return "ok"

print(check_tags("<b>x</b><p><i>y"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"check the leftovers\"\n    expect_output(\"unclosed <i>\")")

### Puzzle: Evaluating postfix

Arrange the lines into a postfix evaluator. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
# Answer without running code (write postfix as a string with single spaces).

p = ___   # "5 - 3 + 1" in postfix
q = ___   # "5 - ( 3 + 1 )" in postfix
r = ___   # the value of "2 3 4 * +"
s = ___   # the value of "2 3 ^ 2 ^" (where ^ is power)
t = ___   # "2 ^ 3 ^ 2" in postfix, with ^ right-associative

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"5 3 - 1 +\" and q == \"5 3 1 + -\" and r == 14, \"p, q and r\"\n    assert s == 64 and t == \"2 3 2 ^ ^\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a robust calculator

Write `calculate(text)`: work out an expression with decimal numbers, `+ - * /`, brackets and spaces, using a tokeniser, shunting-yard and a postfix evaluator. Two extras:

- **Unary minus**: a `-` at the start, after `(` or after another operator, makes the next number (or bracket) negative: `-3 + 5` is 2, `2 * -(1 + 1)` is -4. Hint: when you meet a unary minus, put a special operator `"neg"` with the **highest** precedence, right-associative, that pops **one** operand.
- **Errors**: raise `ValueError` (with any message) for anything malformed: unknown characters, a number like `1.2.3`, mismatched brackets, an operator with a missing operand (`"2 +"`, `"* 3"`), two numbers in a row (`"2 3"`), or empty input. Division by zero raises `ZeroDivisionError` as usual.

Return a `float`. Don't use `eval`.

In [ ]:
def calculate(text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"values\"\n    cases = {\"2*(3+4.5)\": 15.0, \"10 - 2 - 3\": 5.0, \"7 / 2 + 1\": 4.5, \"-3 + 5\": 2.0, \"2 * -(1 + 1)\": -4.0, \"--2\": 2.0, \"1 - -1\": 2.0,\n             \"((1))\": 1.0, \"8 / 4 / 2\": 1.0, \"2 + 3 * 4 - 5\": 9.0, \"-(2)\": -2.0, \"3 * (2 - (1 + 1))\": 0.0}\n    for text, want in cases.items():\n        got = calculate(text)\n        assert isinstance(got, float) and abs(got - want) < 1e-9, (text, got)\ndef test_errors():\n    \"errors\"\n    for bad in [\"2 +\", \"* 3\", \"2 3\", \"\", \"   \", \"(1 + 2\", \"1 + 2)\", \"1.2.3\", \"2 $ 3\", \"()\", \"4 (2)\"]:\n        try:\n            calculate(bad)\n            assert False, f\"{bad!r} should raise ValueError\"\n        except ValueError:\n            pass\n    try:\n        calculate(\"1 / (2 - 2)\")\n        assert False\n    except ZeroDivisionError:\n        pass\n    assert \"eval(\" not in source")

## Warm-up

`3 4 + 2 *` is `(3 + 4) * 2` written in **postfix**: each operator comes after its two operands, so no brackets or precedence rules are needed, and one stack works it out. Notice `b` is popped before `a`: the second operand is on top.

## Nesting, with positions

The stack pattern for nesting: push each opener (with its position, so errors can say where); for each closer, the top must be its partner; at the end the stack must be empty. A dictionary of partners keeps it tidy:

The same shape checks anything nested: HTML or XML tags (`<b>` must close with `</b>`), `begin` and `end` blocks, indentation levels. Only what counts as an opener and a closer changes.

## Postfix

In **postfix** (also called reverse Polish notation, after the Polish logician Jan Łukasiewicz, whose "Polish notation" put operators **before** operands), operators come after their operands. Ordinary notation, with operators between operands, is **infix**:

| Infix | Postfix |
|---|---|
| `3 + 4` | `3 4 +` |
| `(3 + 4) * 2` | `3 4 + 2 *` |
| `3 + 4 * 2` | `3 4 2 * +` |
| `10 - 2 - 3` | `10 2 - 3 -` |

Evaluating postfix needs one stack and one rule: a number is pushed; an operator pops two numbers (the **second** operand first), applies itself, and pushes the result. At the end, exactly one number should remain. (The evaluators here pick the operation with a conditional expression. A dictionary of results such as `{"+": a + b, "/": a / b}[t]` looks neater, but it works out **every** result before choosing one, so `5 - 0` would crash dividing by zero.) Postfix needs no brackets because the order of the tokens already says what happens first. Some calculators and programming languages (and the bytecode Python itself runs) work this way.

## Shunting-yard

People write infix, so a calculator must convert infix to postfix first. Dijkstra's **shunting-yard algorithm** (1961, named after a railway yard where carriages are sorted using a side track) does it with one stack of **operators**:

- A **number** goes straight to the output.
- An **operator** first pops to the output every operator on the stack that should happen **before** it: those with higher precedence, or equal precedence if the new one is left-associative. Then it's pushed.
- `(` is pushed. `)` pops operators to the output until the matching `(`, which is thrown away.
- At the end, pop every remaining operator to the output.

**Precedence** says `*` binds tighter than `+`. **Associativity** says what happens with equal precedence: `10 - 2 - 3` means `(10 - 2) - 3` (left-associative), but powers usually group to the right: `2 ^ 3 ^ 2` means `2 ^ (3 ^ 2)`.

Each token is pushed and popped at most once, so the conversion is O(n).

## A calculator

A calculator is three small parts, each easy to test on its own (Week 34's single responsibility):

1. A **tokeniser** turns `"2*(3+4.5)"` into `["2", "*", "(", "3", "+", "4.5", ")"]`, handling numbers of several digits and spaces.
2. **Shunting-yard** turns the tokens into postfix.
3. A **postfix evaluator** works it out.

This one has no error checking beyond the tokeniser (try `"2 +"`); today's challenge makes it robust, and adds negative numbers.

## Undo, properly

Week 36 built undo and redo with two stacks of commands: do a command, push it on the undo stack and clear the redo stack; undo pops from undo, reverses it, and pushes it on redo. Real editors add two refinements.

**Limited history.** Keeping every change for ever uses ever more memory. A `deque(maxlen=n)` as the undo stack keeps the most recent n steps and silently forgets the oldest (Day 2). The redo stack never needs a limit: it can't hold more than the undo stack once did.

**Grouped steps.** Undo after typing "hello" should remove the word, not one letter. Editors **coalesce** consecutive actions of the same kind into one undo step: typing letters joins the previous typing step, until something else happens (a space, a deletion, moving the cursor, or a pause).

Storing a copy of the whole text before each step is the simplest design, and fine for short texts; for long documents, store commands (Week 36) that know how to reverse themselves instead.

## Practice

## Quiz

## Daily challenge

## Before you go

Run the shunting-yard example by hand on `1 + 2 * ( 3 - 4 ) / 5`, writing the output and the operator stack after each token. Tomorrow: circular buffers, a fixed block of slots used as a queue that never needs to move anything.